## Course map

| Notebook | Main focus |
|---|---|
| Beginner | DX-RT role, device checks, DXNN inspection, and CLI inference |
| Intermediate | Python and C++ APIs, synchronous/asynchronous execution, batch, and buffers |
| Advanced | Resource binding, profiling, monitoring, multi-input/memory loading, and release validation |

Complete the Beginner tutorial first if <code>dxparse</code>, <code>dxrun</code>, and the DXNN tensor contract are unfamiliar.


# DX-RT Tutorial 2: Intermediate

This notebook moves from CLI validation to application-level inference with the DX-RT Python and C++ APIs.

## Learning objectives

By the end of this tutorial, you will be able to:

- install the matching prebuilt <code>dx_engine</code> wheel into the Jupyter kernel,
- read tensor metadata and allocate inputs with the required shape and dtype,
- implement synchronous inference,
- implement asynchronous inference with job IDs and <code>wait()</code>,
- explain callback and buffer-lifetime rules,
- group independent samples with the batch API,
- use <code>InferenceOption.buffer_count</code> deliberately,
- build a minimal C++14 application with <code>dxrt_cxx_api.h</code>, and
- select an execution mode from latency, throughput, and ownership requirements.

All generated Python, C++, build, and report files remain under <code>&lt;dx-tutorials&gt;/notebooks/T06-DX-Runtime/workspace</code>.


## 1. Initialize the tutorial workspace


In [ ]:
from pathlib import Path
import importlib.metadata
import os
import shlex
import sys
import time

import numpy as np

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")

%run "$root_path/tutorial_paths.py"

T06_DIR = TUTORIAL_ROOT / "notebooks" / "T06-DX-Runtime"
WORK_DIR = T06_DIR / "workspace"
MODEL_DIR = WORK_DIR / "models"
REPORT_DIR = WORK_DIR / "reports"
PYTHON_DIR = WORK_DIR / "python"
CPP_DIR = WORK_DIR / "cpp"
CPP_BUILD_DIR = CPP_DIR / "build"

for path in (WORK_DIR, MODEL_DIR, REPORT_DIR, PYTHON_DIR, CPP_DIR):
    path.mkdir(parents=True, exist_ok=True)

MODEL_SOURCE = DX_WORKSPACE_DIR / "res" / "models" / "resnet50_224x224.dxnn"
if not MODEL_SOURCE.is_file():
    command = (
        f"cd {shlex.quote(str(DX_APP_DIR))} && "
        "bash setup.sh --models resnet50 --no-force"
    )
    raise FileNotFoundError(
        f"Required model was not found: {MODEL_SOURCE}\n"
        f"Run this command in a terminal, then rerun this cell:\n{command}"
    )

MODEL_PATH = MODEL_DIR / MODEL_SOURCE.name
if not MODEL_PATH.exists():
    MODEL_PATH.symlink_to(MODEL_SOURCE)

os.chdir(WORK_DIR)
print(f"Kernel Python : {sys.executable}")
print(f"Workspace     : {WORK_DIR}")
print(f"Model         : {MODEL_PATH}")


## 2. Install the Python binding in the T06 workspace

The DX-RT Debian package provides version-specific wheels under <code>/usr/share/libdxrt-bin/python</code>. A wheel built for Python 3.12 cannot be imported by a Python 3.13 kernel, so this notebook selects the wheel whose <code>cpXY</code> tag matches the running kernel.

To keep every generated file inside T06, this tutorial creates a small dedicated environment under <code>workspace/.venv-dxrt</code> and exposes only its site-packages directory to the current kernel. The commands are equivalent to:

~~~bash
uv venv --python <current-jupyter-python> <T06>/workspace/.venv-dxrt
uv pip install --python <T06>/workspace/.venv-dxrt/bin/python \
  /usr/share/libdxrt-bin/python/dx_engine-<version>-cp<XY>-*.whl
~~~

This does not modify the shared Jupyter environment and does not rebuild DX-RT. Re-running the cells skips creation and installation when the local environment already contains the matching wheel.


In [ ]:
WHEEL_DIR = Path("/usr/share/libdxrt-bin/python")
python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
matching_wheels = sorted(WHEEL_DIR.glob(f"dx_engine-*-{python_tag}-{python_tag}-*.whl"))

if not matching_wheels:
    raise FileNotFoundError(
        f"No dx_engine wheel for {python_tag} was found under {WHEEL_DIR}. "
        "Install the current libdxrt-bin package first."
    )

DX_ENGINE_WHEEL = matching_wheels[-1]
DXRT_PYTHON_ENV = WORK_DIR / ".venv-dxrt"
DXRT_PYTHON = DXRT_PYTHON_ENV / "bin" / "python"
DXRT_SITE_PACKAGES = (
    DXRT_PYTHON_ENV
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
)
wheel_version = DX_ENGINE_WHEEL.name.split("-")[1]

print(f"Selected wheel : {DX_ENGINE_WHEEL}")
print(f"Local Python   : {DXRT_PYTHON}")


In [ ]:
if not DXRT_PYTHON.is_file():
    !uv venv --python "{sys.executable}" "{DXRT_PYTHON_ENV}"
else:
    print(f"Local environment already exists; skipping creation: {DXRT_PYTHON_ENV}")

installed_dx_engine = next(
    (
        distribution.version
        for distribution in importlib.metadata.distributions(
            path=[str(DXRT_SITE_PACKAGES)]
        )
        if distribution.metadata.get("Name", "").lower().replace("_", "-")
        == "dx-engine"
    ),
    None,
)

if installed_dx_engine != wheel_version:
    !uv pip install --python "{DXRT_PYTHON}" "{DX_ENGINE_WHEEL}"
else:
    print(f"dx-engine {installed_dx_engine} already matches; skipping installation.")


In [ ]:
if not DXRT_SITE_PACKAGES.is_dir():
    raise FileNotFoundError(f"Local site-packages was not created: {DXRT_SITE_PACKAGES}")
sys.path.insert(0, str(DXRT_SITE_PACKAGES))

from dx_engine import InferenceEngine, InferenceOption, __version__ as dx_engine_version

print(f"dx_engine version: {dx_engine_version}")
print(f"dx_engine source : {Path(sys.modules['dx_engine'].__file__).resolve()}")


## 3. Read tensor metadata before allocating memory

Do not infer the input shape or dtype from a model name. Ask the engine for its contract. DX-RT v3.4 validates NumPy dtypes to prevent undefined behavior.

The helper below uses <code>np.empty(...)</code> and then fills the buffer. This forces real writable pages to be allocated, avoiding copy-on-write zero-page problems during DMA pinning.


In [ ]:
def make_input(tensor_info: dict, fill_value: int = 0) -> np.ndarray:
    array = np.empty(tensor_info["shape"], dtype=tensor_info["dtype"])
    array.fill(fill_value)
    return np.ascontiguousarray(array)

with InferenceEngine(str(MODEL_PATH)) as ie:
    input_info = ie.get_input_tensors_info()
    output_info = ie.get_output_tensors_info()

print("Input tensors:")
for tensor in input_info:
    print(tensor)

print("\nOutput tensors:")
for tensor in output_info:
    print(tensor)


### 3.1 Tensor ownership checklist

| Check | Safe practice |
|---|---|
| Shape | Allocate from <code>get_input_tensors_info()</code> |
| Dtype | Use the returned NumPy dtype exactly |
| Layout | Match the compiled model's visible layout |
| Contiguity | Pass C-contiguous arrays |
| Lifetime | Keep asynchronous inputs alive until completion |
| Output scope | Copy callback output only when it must outlive the callback |


## 4. Synchronous inference

<code>run()</code> blocks the calling thread until the result is ready. It is the clearest starting point for a sequential or latency-oriented application.

<img src="assets/dx-rt-execution-modes.svg" style="max-width: 1100px; width: 100%;" alt="Synchronous, asynchronous, and batch execution modes">


In [ ]:
with InferenceEngine(str(MODEL_PATH)) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    input_tensor = make_input(tensor_info)

    start = time.perf_counter()
    outputs = ie.run([input_tensor])
    elapsed_ms = (time.perf_counter() - start) * 1000

    print(f"Output tensor count : {len(outputs)}")
    for index, output in enumerate(outputs):
        print(f"output[{index}] shape={output.shape}, dtype={output.dtype}, bytes={output.nbytes}")
    print(f"Host-observed time  : {elapsed_ms:.3f} ms")
    print(f"DX-RT latency       : {ie.get_latency() / 1000:.3f} ms")
    print(f"NPU inference time  : {ie.get_npu_inference_time() / 1000:.3f} ms")


The three timings answer different questions:

- **Host-observed time** includes the Python call and host-side overhead around it.
- **DX-RT latency** is the runtime's latest request latency.
- **NPU inference time** covers NPU execution and is only one part of end-to-end latency.

Do not label NPU time as camera-to-display latency.


## 5. Asynchronous inference with job IDs

<code>run_async()</code> returns a job ID. <code>wait(job_id)</code> later returns the matching output. This allows the application to overlap submission, NPU execution, and other work.

<img src="assets/dx-rt-buffer-lifecycle.svg" style="max-width: 1100px; width: 100%;" alt="Asynchronous buffer ownership lifecycle">


In [ ]:
ASYNC_JOBS = 8

option = InferenceOption()
option.buffer_count = 4

with InferenceEngine(str(MODEL_PATH), option) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    async_inputs = [make_input(tensor_info, index) for index in range(ASYNC_JOBS)]

    start = time.perf_counter()
    job_ids = [ie.run_async([array]) for array in async_inputs]
    async_outputs = [ie.wait(job_id) for job_id in job_ids]
    elapsed_s = time.perf_counter() - start

print(f"Submitted job IDs : {job_ids}")
print(f"Completed outputs : {len(async_outputs)}")
print(f"Throughput         : {ASYNC_JOBS / elapsed_s:.2f} requests/s")


### 5.1 Wait versus callback

| Completion style | Strength | Main responsibility |
|---|---|---|
| <code>run_async()</code> + <code>wait(job_id)</code> | Explicit request/result matching | Store job IDs and wait from an appropriate thread |
| Registered callback | Low-latency completion handling | Keep the callback fast and thread-safe |
| <code>run()</code> | Simplest control flow | Accept a blocked calling thread |

Callback outputs are valid only inside the callback scope. If downstream work must retain them, copy the required data inside the callback and move heavy work to another queue.


## 6. Batch API

DX-RT batch execution groups several independent samples and schedules them asynchronously inside the runtime. It does **not** change the compiled model's batch dimension; the DXNN model still normally uses batch size 1. The current API uses `run()` with batch-formatted inputs and explicit output buffers; `run_batch()` is retained only as a deprecated compatibility wrapper.

The nested Python form is:

~~~python
[
    [sample_0_input_0],
    [sample_1_input_0],
    [sample_2_input_0],
]
~~~


In [ ]:
BATCH_SIZE = 4

with InferenceEngine(str(MODEL_PATH)) as ie:
    tensor_info = ie.get_input_tensors_info()[0]
    output_info = ie.get_output_tensors_info()
    batch_inputs = [[make_input(tensor_info, index)] for index in range(BATCH_SIZE)]
    batch_output_buffers = [
        [
            np.empty(info["shape"], dtype=info["dtype"])
            for info in output_info
        ]
        for _ in range(BATCH_SIZE)
    ]

    start = time.perf_counter()
    batch_outputs = ie.run(
        batch_inputs,
        output_buffers=batch_output_buffers,
    )
    elapsed_s = time.perf_counter() - start

print(f"Batch samples      : {len(batch_outputs)}")
print(f"Outputs per sample : {[len(sample) for sample in batch_outputs]}")
print(f"Effective rate     : {BATCH_SIZE / elapsed_s:.2f} samples/s")


## 7. Buffer count is pipeline capacity

<code>InferenceOption.buffer_count</code> controls the number of internal inference buffers. A larger value can permit more in-flight work, but it also consumes more memory and may stop helping after the pipeline is full.

| Too small | Balanced | Too large |
|---|---|---|
| Submitter waits for buffers | Enough work to keep the NPU busy | Extra memory with little gain |
| Lower throughput is possible | Stable throughput and bounded memory | Longer queues can increase latency |

Measure representative values instead of assuming that the largest value is best. The Advanced tutorial performs a controlled buffer-count experiment.


## 8. Build a minimal C++14 application

DX-RT v3.4 provides a stable C ABI and a header-only C++14 wrapper. New C++ code should include only:

~~~cpp
#include <dxrt/dxrt_cxx_api.h>
~~~

Do not include <code>dxrt_api.h</code> and <code>dxrt_cxx_api.h</code> in the same translation unit.

The next two cells create source files only under <code>T06-DX-Runtime/workspace/cpp</code>.


In [ ]:
%%writefile {CPP_DIR / "main.cpp"}
#include <dxrt/dxrt_cxx_api.h>

#include <chrono>
#include <cstdint>
#include <exception>
#include <iostream>
#include <string>
#include <vector>

int main(int argc, char* argv[])
{
    if (argc != 2) {
        std::cerr << "Usage: dxrt_sync <model.dxnn>\n";
        return 2;
    }

    try {
        dxrt::InferenceEngine engine(argv[1]);
        std::vector<std::uint8_t> input(engine.GetInputSize(), 0);

        const auto start = std::chrono::steady_clock::now();
        const auto outputs = engine.Run(input.data());
        const auto end = std::chrono::steady_clock::now();

        const double elapsed_ms =
            std::chrono::duration<double, std::milli>(end - start).count();

        std::cout << "Input bytes   : " << input.size() << '\n';
        std::cout << "Output tensors: " << outputs.size() << '\n';
        for (std::size_t i = 0; i < outputs.size(); ++i) {
            std::cout << "  [" << i << "] " << outputs[i]->name()
                      << ", bytes=" << outputs[i]->size_in_bytes() << '\n';
        }
        std::cout << "Host time     : " << elapsed_ms << " ms\n";
    } catch (const std::exception& error) {
        std::cerr << "DX-RT error: " << error.what() << '\n';
        return 1;
    }
    return 0;
}


In [ ]:
%%writefile {CPP_DIR / "CMakeLists.txt"}
cmake_minimum_required(VERSION 3.16)
project(dxrt_sync LANGUAGES CXX)

find_path(DXRT_INCLUDE_DIR
    NAMES dxrt/dxrt_cxx_api.h
    PATHS /usr/local/include
    REQUIRED
)
find_library(DXRT_LIBRARY
    NAMES dxrt
    PATHS /usr/local/lib /usr/lib
    REQUIRED
)

add_executable(dxrt_sync main.cpp)
target_compile_features(dxrt_sync PRIVATE cxx_std_14)
target_include_directories(dxrt_sync PRIVATE "${DXRT_INCLUDE_DIR}")
target_link_libraries(dxrt_sync PRIVATE "${DXRT_LIBRARY}" pthread)


### 8.1 Configure and build

These notebook cells run the same commands you would enter in a terminal:

~~~bash
cmake -S <T06>/workspace/cpp -B <T06>/workspace/cpp/build \
      -DCMAKE_BUILD_TYPE=Release
cmake --build <T06>/workspace/cpp/build --parallel
~~~

The build directory stays inside T06.


In [ ]:
!cmake -S "{CPP_DIR}" -B "{CPP_BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release
!cmake --build "{CPP_BUILD_DIR}" --parallel


### 8.2 Run the C++ application

Equivalent terminal command:

~~~bash
<T06>/workspace/cpp/build/dxrt_sync \
  <T06>/workspace/models/resnet50_224x224.dxnn
~~~


In [ ]:
!"{CPP_BUILD_DIR / 'dxrt_sync'}" "{MODEL_PATH}"


## 9. Choose an execution style

| Requirement | Recommended starting point | Reason |
|---|---|---|
| Simplest sequential flow | Synchronous <code>run()</code> | Clear ownership and error handling |
| Lowest single-request latency study | Synchronous <code>run()</code> | No intentional queue depth |
| Higher streaming throughput | Async + <code>wait()</code> or callback | Overlaps independent pipeline work |
| Several independent samples at once | Batch API | Groups submission and completion |
| Tight integration and low Python overhead | C++ API | Direct C++14 application control |
| Existing Python pipeline | Python API | Fast integration with NumPy data |

The correct choice depends on the complete application. A higher inference FPS can still produce worse user-visible latency if queues become too deep.


## 10. Summary

### 10.1 API workflow completed

**Read tensor metadata**  
→ **Allocate exact dtype and shape**  
→ **Run synchronously**  
→ **Submit and wait asynchronously**  
→ **Group independent samples**  
→ **Build the same flow in C++**

<img src="assets/dx-rt-execution-modes.svg" style="max-width: 1000px; width: 100%;" alt="DX-RT execution modes">

### 10.2 Execution dashboard

| Mode | Submission | Completion | Best first metric |
|---|---|---|---|
| Sync | One request | Return from <code>run()</code> | Request latency |
| Async + Wait | Several job IDs | <code>wait(job_id)</code> | Throughput and queue delay |
| Callback | Several requests | Runtime callback | Callback cost and throughput |
| Batch | Group of samples | Group of outputs | Effective samples/s |
| C++ | Same runtime concepts | C++ tensors | End-to-end application cost |

### 10.3 Completion checklist

- [x] Installed the wheel matching the Jupyter Python ABI
- [x] Allocated a contiguous tensor from runtime metadata
- [x] Measured synchronous host, runtime, and NPU timings
- [x] Preserved asynchronous input lifetime through <code>wait()</code>
- [x] Used the batch API without changing model batch size
- [x] Explained the memory/throughput tradeoff of <code>buffer_count</code>
- [x] Built and ran a C++14 DX-RT application
- [ ] Replace dummy input with product preprocessing and validate outputs

> **Remember:** buffer ownership is part of correctness. Performance tuning comes after shape, dtype, layout, lifetime, and output mapping are verified.

### Next step

Continue with the **Advanced tutorial** to bind devices and cores, profile individual stages, monitor device health, test memory-loaded and multi-input models, and build reproducible release evidence.
